In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 וה-Qiskit SDK

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
```
</details>

ה-Qiskit SDK מספק כלים להמרה בין ייצוגי OpenQASM של תוכניות קוונטיות לבין המחלקה [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## ייבוא תוכנית OpenQASM 2 אל Qiskit
שתי פונקציות מייבאות תוכניות OpenQASM 2 אל Qiskit.
אלו הן [`qasm2.load()`](../api/qiskit/qasm2#load), שמקבלת שם קובץ, ו-[`qasm2.loads()`](../api/qiskit/qasm2#loads), שמקבלת את תוכנית OpenQASM 2 כמחרוזת.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

ראה את [ה-API של OpenQASM 2 ב-Qiskit](https://docs.quantum.ibm.com/api/qiskit/qasm2) לפרטים נוספים.

### ייבוא תוכניות פשוטות
עבור רוב תוכניות OpenQASM 2, אתה יכול פשוט להשתמש ב-`qasm2.load` וב-`qasm2.loads` עם ארגומנט יחיד.

#### דוגמה: ייבוא תוכנית OpenQASM 2 כמחרוזת
השתמש ב-`qasm2.loads()` כדי לייבא תוכנית OpenQASM 2 כמחרוזת אל תוך QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### דוגמה: ייבוא תוכנית OpenQASM 2 מקובץ
השתמש ב-`load()` כדי לייבא תוכנית OpenQASM 2 מקובץ אל תוך QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### קישור שערי OpenQASM 2 עם שערי Qiskit
כברירת מחדל, מייבא OpenQASM 2 של Qiskit מתייחס לקובץ ה-include `"qelib1.inc"` כספריית סטנדרט *de facto*.
המייבא מתייחס לקובץ זה כמכיל בדיוק את השערים המתוארים בו ב[מאמר המקורי שהגדיר את OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit ישתמש בשערים המובנים ב[ספריית ה-Circuit](../api/qiskit/circuit_library) כדי לייצג את השערים ב-`"qelib1.inc"`.
שערים שמוגדרים בתוכנית באמצעות הצהרות `gate` ידניות של OpenQASM 2 ייבנו כברירת מחדל כתת-מחלקות מותאמות אישית של [Qiskit `Gate`](../api/qiskit/qiskit.circuit.Gate).

אתה יכול להורות למייבא להשתמש במחלקות [`Gate`](../api/qiskit/qiskit.circuit.Gate) ספציפיות עבור הצהרות `gate` נתונות שהוא נתקל בהן.
אתה יכול גם להשתמש במנגנון זה כדי להתייחס לשמות שערים נוספים כ"מובנים", כלומר, שאינם דורשים הגדרה מפורשת.
אם תציין אילו מחלקות שערים להשתמש בהן עבור הצהרות `gate` שמחוץ ל-`"qelib1.inc"`, ה-Circuit שיתקבל יהיה בדרך כלל יעיל יותר לעבודה.

> **Warning:** החל מ-Qiskit SDK גרסה 1.0, *המייצא* של Qiskit ל-OpenQASM 2 (ראה [ייצוא Circuit של Qiskit אל OpenQASM 2](#qasm2-export)) עדיין מתנהג כאילו ל-`"qelib1.inc"` יש יותר שערים ממה שיש לו בפועל.
> פירוש הדבר הוא שהגדרות ברירת המחדל של המייבא עשויות שלא להיות מסוגלות לייבא תוכנית שיוצאה על-ידי המייצא שלנו.
> ראה [את הדוגמה הספציפית לעבודה עם המייצא הישן](#qasm2-import-legacy) כדי לפתור בעיה זו.
> 
> אי-התאמה זו היא התנהגות ישנה של Qiskit, ו[היא תיפתר בגרסה עתידית של Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

כדי להעביר מידע על הוראה מותאמת אישית למייבא OpenQASM 2, השתמש ב[מחלקה `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
יש לה ארבעה פרטי מידע נדרשים, לפי הסדר:

* ה**שם** של השער, כפי שמשמש בתוכנית OpenQASM 2
* **מספר פרמטרי הזווית** שהשער מקבל
* **מספר ה-Qubit** שעליהם השער פועל
* מחלקת **הבנאי** בפייתון או הפונקציה עבור השער, שמקבלת את פרמטרי השער (אך לא את ה-Qubit) כארגומנטים נפרדים

אם המייבא נתקל בהגדרת `gate` שתואמת להוראה מותאמת אישית נתונה, הוא ישתמש במידע המותאם אישית הזה כדי לשחזר את אובייקט השער.
אם נתקלים בהצהרת `gate` שתואמת ל`name` של הוראה מותאמת אישית, אך אינה תואמת גם את מספר הפרמטרים וגם את מספר ה-Qubit, המייבא יזרוק [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) כדי לציין את אי-ההתאמה בין המידע שסופק לבין התוכנית.

בנוסף, ניתן לאפשר אופציונלית ארגומנט חמישי `builtin` על-ידי הגדרתו ל-`True` כדי להפוך את השער לזמין אוטומטית בתוכנית OpenQASM 2, גם אם הוא לא מוגדר במפורש.
אם המייבא נתקל בהגדרת `gate` מפורשת עבור הוראה מותאמת אישית מובנית, הוא יקבל אותה בשקט.
כבעבר, אם הגדרה מפורשת של אותו שם אינה תואמת להוראה המותאמת אישית שסופקה, יוזרק [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
זה שימושי לתאימות עם מייצאי OpenQASM 2 ישנים יותר, ועם פלטפורמות קוונטיות מסוימות אחרות שמתייחסות ל"שערי הבסיס" של חומרתן כהוראות מובנות.

Qiskit מספקת תכונת נתונים לעבודה עם תוכניות OpenQASM 2 שנוצרו על-ידי גרסאות ישנות של [יכולות ייצוא OpenQASM 2 של Qiskit](#qasm2-export).
זהו [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), שניתן לספק אותו כארגומנט `custom_instructions` ל-[`qasm2.load()`](../api/qiskit/qasm2#load) ול-[`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### דוגמה: ייבוא תוכנית שנוצרה על-ידי המייצא הישן של Qiskit
תוכנית OpenQASM 2 זו משתמשת בשערים שאינם בגרסה המקורית של `"qelib1.inc"` מבלי להצהיר עליהם, אך הם שערים סטנדרטיים בספריית Qiskit.
אתה יכול להשתמש ב-[`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) כדי בקלות להורות למייבא להשתמש באותה קבוצת שערים שמייצא OpenQASM 2 של Qiskit השתמש בה בעבר.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### דוגמה: שימוש במחלקת Gate ספציפית בעת ייבוא תוכנית OpenQASM 2
Qiskit אינה יכולה, בדרך כלל, לאמת אם ההגדרה בהצהרת `gate` של OpenQASM 2 תואמת בדיוק לשער ספריית סטנדרט של Qiskit.
במקום זאת, Qiskit בוחרת שער מותאם אישית באמצעות ההגדרה המדויקת שסופקה.
זה יכול להיות פחות יעיל מאשר שימוש באחד מהשערים הסטנדרטיים המובנים, או בשער מותאם אישית שהמשתמש הגדיר.
אתה יכול להגדיר ידנית הצהרות `gate` עם מחלקות ספציפיות.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### דוגמה: הגדרת Gate מובנה חדש בתוכנית OpenQASM 2
אם מוגדר הארגומנט `builtin=True`, שער מותאם אישית אינו צריך להיות בעל הגדרה משויכת.